# Topic 1 — Tokens

Understanding how text becomes tokens before an LLM processes it.

This notebook implements the core ideas behind tokenization from scratch,
then compares the results with a real tokenizer.

## Learning Objectives

By the end of this notebook, I should be able to:

- explain why LLMs do not directly consume words or characters
- explain the path from text to bytes to BPE tokens to vocabulary IDs
- implement a simple tokenizer from scratch
- implement the core BPE merge mechanism
- encode text into token IDs and decode IDs back to text
- explain why common text tends to require fewer tokens
- identify cases where tokenization behaves unexpectedly
- count tokens using a production tokenizer
- predict token counts before verifying them

## What is a token?

Text
 ↓
Tokenizer
 ↓
Tokens
 ↓
Integer IDs
 ↓
LLM

A token is not necessarily a word.

A token can represent:
- a complete word
- part of a word
- punctuation
- whitespace
- a combination of common characters/bytes

In [ ]:
examples = [
    'hello',
    'hello world',
    'machine learning',
    'unbelievable',
    '12345',
    'hello!'
]
for s in examples:
    print(f'{s!r}: {len(s)} characters')

## Before Building a Tokenizer

If an LLM simply consumed one character at a time:

- What would happen to sequence length?
- What would happen to common words?
- What would happen to repeated text?
- What would happen to non-English text?

## Why start with characters?

character tokenizer
        ↓
simple to understand
        ↓
but inefficient for language
        ↓
motivation for subword/byte-level tokenization

In [ ]:
sentence = 'Tokens are the units LLMs actually process.'
for i, ch in enumerate(sentence):
    print(f'{i:>2}: {ch!r}')
print(f'Total characters: {len(sentence)}')

In [ ]:
text = 'Tokens are the units LLMs actually process.'
chars = sorted(set(text))
char_to_id = {ch: idx for idx, ch in enumerate(chars)}
id_to_char = {idx: ch for ch, idx in char_to_id.items()}
print('Character vocabulary:')
for ch, idx in sorted(char_to_id.items(), key=lambda x: x[1]):
    print(f'{ch!r}: {idx}')
print('\nReverse lookup sample:')
print(id_to_char[0])

In [ ]:
def char_encode(text, char_to_id):
    return [char_to_id[ch] for ch in text]

text = 'hello'
char_to_id = {ch: idx for idx, ch in enumerate(sorted(set(text)))}
print(char_encode(text, char_to_id))

In [ ]:
def char_decode(ids, id_to_char):
    return ''.join(id_to_char[i] for i in ids)

text = 'hello'
char_to_id = {ch: idx for idx, ch in enumerate(sorted(set(text)))}
id_to_char = {idx: ch for ch, idx in char_to_id.items()}
encoded = char_encode(text, char_to_id)
decoded = char_decode(encoded, id_to_char)
print(encoded)
print(decoded)
print(decoded == text)

In [ ]:
samples = [
    'hello',
    'hello world',
    'machine learning',
    'unbelievable',
    '12345',
    'hello!'
]
for sample in samples:
    print(f'{sample!r}: chars={len(sample)}')
print('Character tokenization is simple, but it is often inefficient for ordinary language.')

## Why bytes?

characters
   ↓
UTF-8 encoding
   ↓
bytes
   ↓
BPE merges
   ↓
tokens

In [ ]:
s = 'hello'
encoded = s.encode('utf-8')
print('string:', s)
print('utf-8 bytes:', encoded)
print('byte values:', list(encoded))

In [ ]:
samples = {
    'English': 'hello world',
    'Telugu': 'హలో ప్రపంచం',
    'Hindi': 'नमस्ते दुनिया',
    'Japanese': 'こんにちは世界',
    'emoji': '🚀✨'
}
for label, text in samples.items():
    b = text.encode('utf-8')
    print(f'{label}: chars={len(text)}, bytes={len(b)}')
    print(f'  bytes={list(b)}')

### Observation

Why can one visible character occupy multiple UTF-8 bytes?

Because UTF-8 is designed to represent the full Unicode character set using a variable-length encoding. Common ASCII characters use 1 byte, while many non-ASCII characters use 2–4 bytes. This matters because byte-level tokenizers can work across scripts without needing a separate character vocabulary for every language.

## The BPE idea

Raw bytes

a a a b d a a a b a c

        ↓ find frequent pair

a a → X

        ↓

X X b d X X b a c

        ↓ repeat

This is the core intuition behind BPE: merge the most common adjacent pieces, repeatedly.

In [ ]:
toy = 'aaabdaaabac'
print('toy text:', toy)
print('length:', len(toy))

In [ ]:
from collections import Counter

def pair_counts(sequence):
    counts = Counter()
    for left, right in zip(sequence, sequence[1:]):
        counts[(left, right)] += 1
    return counts

sequence = [97, 97, 97, 98, 100, 97, 97, 97, 98, 97, 99]
counts = pair_counts(sequence)
print('Adjacent pairs and counts:')
for pair, count in counts.most_common():
    print(pair, count)

In [ ]:
from collections import Counter

def best_pair(sequence):
    counts = Counter()
    for left, right in zip(sequence, sequence[1:]):
        counts[(left, right)] += 1
    if not counts:
        return None, 0
    return max(counts.items(), key=lambda item: (item[1], item[0]))

def merge_pair(sequence, pair):
    merged = []
    i = 0
    while i < len(sequence):
        if i < len(sequence) - 1 and (sequence[i], sequence[i + 1]) == pair:
            merged.append(pair)
            i += 2
        else:
            merged.append(sequence[i])
            i += 1
    return merged

sequence = [97, 97, 97, 98, 100, 97, 97, 97, 98, 97, 99]
pair, frequency = best_pair(sequence)
print('Before:', sequence)
sequence = merge_pair(sequence, pair)
print('After:', sequence)
print('Merged pair:', pair, 'frequency=', frequency)

In [ ]:
from collections import Counter

def pair_counts(sequence):
    counts = Counter()
    for left, right in zip(sequence, sequence[1:]):
        counts[(left, right)] += 1
    return counts

def best_pair(sequence):
    counts = pair_counts(sequence)
    if not counts:
        return None, 0
    return max(counts.items(), key=lambda item: (item[1], item[0]))

def merge_pair(sequence, pair):
    merged = []
    i = 0
    while i < len(sequence):
        if i < len(sequence) - 1 and (sequence[i], sequence[i + 1]) == pair:
            merged.append(pair)
            i += 2
        else:
            merged.append(sequence[i])
            i += 1
    return merged

sequence = [97, 97, 97, 98, 100, 97, 97, 97, 98, 97, 99]
history = []
for step in range(6):
    pair, frequency = best_pair(sequence)
    if pair is None:
        break
    history.append((step, pair, frequency, sequence.copy()))
    sequence = merge_pair(sequence, pair)
    print(f'Step {step}: pair={pair}, frequency={frequency}')
    print('  before=', history[-1][3])
    print('  after =', sequence)

## Vocabulary and token IDs

Vocabulary
=
mapping between token representation and integer ID

For a byte-level BPE tokenizer:

initial vocabulary
    ↓
256 byte tokens
    ↓
new IDs for learned merges

In [ ]:
token_to_id = {byte: byte for byte in range(256)}
id_to_token = {byte: byte for byte in range(256)}
merged_tokens = [(97, 97), (97, 98), (98, 97)]
next_id = 256
for merged in merged_tokens:
    token_to_id[merged] = next_id
    id_to_token[next_id] = merged
    next_id += 1
print('Sample vocabulary entries:')
for key, value in list(token_to_id.items())[:10]:
    print(key, '->', value)
print('Merged token mapping:', token_to_id[(97, 97)])

In [ ]:
token_to_id = {byte: byte for byte in range(256)}
id_to_token = {byte: byte for byte in range(256)}
for idx in range(0, 10):
    print(f'{idx:>3} -> {id_to_token[idx]}')
print('...')
print('Merged tokens are added after the base 256 bytes.')

## Building the Tokenizer

The tokenizer needs two directions:

encode:
text → token IDs

decode:
token IDs → text

In [ ]:
def encode_simple(text, merge_rules=None):
    sequence = list(text.encode('utf-8'))
    if merge_rules is None:
        return sequence
    for pair in merge_rules:
        merged = []
        i = 0
        while i < len(sequence):
            if i < len(sequence) - 1 and (sequence[i], sequence[i + 1]) == pair:
                merged.append(pair)
                i += 2
            else:
                merged.append(sequence[i])
                i += 1
        sequence = merged
    return sequence

print(encode_simple('hello'))

In [ ]:
def decode_simple(token_sequence):
    flat = []
    for token in token_sequence:
        if isinstance(token, tuple):
            flat.extend(token)
        else:
            flat.append(token)
    return bytes(flat).decode('utf-8')

sequence = [104, 101, 108, 108, 111]
print(decode_simple(sequence))

In [ ]:
samples = ['hello', 'hello world', 'machine learning', 'Telugu: హలో', 'emoji: 🚀']
for sample in samples:
    encoded = list(sample.encode('utf-8'))
    decoded = bytes(encoded).decode('utf-8')
    print(f'{sample!r} -> roundtrip={decoded == sample}')

In [ ]:
text = 'machine learning is useful'
pieces = text.split()
print('Original:', text)
print('Tokens:', pieces)

In [ ]:
samples = [
    'hello world',
    'unbelievable',
    'SolidGoldMagikarp',
    'google.com/search?q=tokenizer'
]
for sample in samples:
    b = sample.encode('utf-8')
    print(f'{sample!r}')
    print(f'  chars={len(sample)} bytes={len(b)}')
    print('  token-like complexity is high when text is rare, long, or mixed with punctuation.')

## Tokenization is not word counting

word count ≠ token count
character count ≠ token count

In [ ]:
examples = [
    'common words',
    'rare words',
    'supercalifragilisticexpialidocious',
    'zzzzzzzzzz',
    'tokenization',
    'XxXxXxXx'
]
for example in examples:
    print(f'{example!r}: length={len(example)}')

In [ ]:
r_examples = ['r', 'rr', 'rrr', 'rrrr', 'rrrrr', 'rrrrrr']
for s in r_examples:
    print(f'{s!r}: chars={len(s)} bytes={len(s.encode('utf-8'))}')

In [ ]:
samples = [
    'hello',
    ' hello',
    'hello ',
    ' hello ',
    'hello  world'
]
for sample in samples:
    print(f'{sample!r}: chars={len(sample)}, bytes={len(sample.encode('utf-8'))}')

In [ ]:
samples = [
    'hello',
    'hello!',
    'hello?',
    'hello,',
    'hello',
    'hello...'
]
for sample in samples:
    print(f'{sample!r}: chars={len(sample)}, bytes={len(sample.encode('utf-8'))}')

In [ ]:
samples = {
    'English': 'hello world',
    'Hindi': 'नमस्ते दुनिया',
    'Telugu': 'హలో ప్రపంచం',
    'Japanese': 'こんにちは世界'
}
for label, text in samples.items():
    print(f'{label}: chars={len(text)}, bytes={len(text.encode('utf-8'))}')

## Why use a real tokenizer now?

Our tokenizer is an educational implementation.

Real production tokenizers contain additional details and model-specific vocabularies.

Now we compare our observations with a real tokenizer.

In [ ]:
try:
    import tiktoken
    print('tiktoken already installed')
except ImportError:
    %pip install tiktoken
    import tiktoken
    print('tiktoken installed')

enc = tiktoken.get_encoding('cl100k_base')
print('encoding loaded')

In [ ]:
enc = tiktoken.get_encoding('cl100k_base')
examples = [
    'hello',
    'hello world',
    'machine learning',
    'supercalifragilisticexpialidocious',
    'Hello, world!',
    'हेलो दुनिया'
]
for text in examples:
    ids = enc.encode(text)
    print(f'{text!r} -> ids={ids} -> count={len(ids)}')

In [ ]:
enc = tiktoken.get_encoding('cl100k_base')
text = 'hello world'
ids = enc.encode(text)
decoded = enc.decode(ids)
print('ids:', ids)
print('decoded:', decoded)
print('roundtrip:', decoded == text)

In [ ]:
enc = tiktoken.get_encoding('cl100k_base')
examples = [
    'machine learning is useful',
    'SolidGoldMagikarp',
    'hello  world',
    'привет мир'
]
for text in examples:
    ids = enc.encode(text)
    pieces = [enc.decode([i]) for i in ids]
    print(f'Text: {text!r}')
    print(f'IDs: {ids}')
    print(f'Pieces: {pieces}')
    print(f'Token count: {len(ids)}')
    print()

## Prediction table

Create a table containing your own predictions.

Columns:

Text
My prediction
Reason

Use examples covering:

- common words
- rare/unusual words
- punctuation
- whitespace
- repeated characters
- different casing
- non-English text
- code-like strings
- long identifiers

Do not run the tokenizer yet.

In [ ]:
enc = tiktoken.get_encoding('cl100k_base')
predictions = [
    ('hello', 1),
    ('hello world', 2),
    ('HELLO WORLD', 3),
    ('hello!', 2),
    ('hello  world', 4),
    ('SolidGoldMagikarp', 5),
    ('привет мир', 4),
    ('rrrrrrrrrrrr', 3),
]
for text, my_prediction in predictions:
    actual = len(enc.encode(text))
    print(f'{text!r}: predicted={my_prediction}, actual={actual}, correct={my_prediction == actual}')

In [ ]:
enc = tiktoken.get_encoding('cl100k_base')
examples = [
    ('hello', 1),
    ('hello world', 2),
    ('hello!', 2),
    ('hello  world', 4),
    ('SolidGoldMagikarp', 5),
    ('привет мир', 4),
]
for text, predicted in examples:
    actual = len(enc.encode(text))
    diff = actual - predicted
    print(f'{text!r}: predicted={predicted}, actual={actual}, diff={diff}')

## Cost

more tokens
    ↓
more input/output token usage
    ↓
potentially higher API cost

## Context limits

context window
=
token budget

So:

more tokens per piece of text
        ↓
fewer pieces of text fit into a fixed token budget

In [ ]:
enc = tiktoken.get_encoding('cl100k_base')
paragraph = 'Tokenization affects cost, context limits, and odd edge cases in language models. This sentence is intentionally simple but demonstrates why token counts are not the same as word counts.'
ids = enc.encode(paragraph)
print('paragraph:', paragraph)
print('chars:', len(paragraph))
print('bytes:', len(paragraph.encode('utf-8')))
print('tokens:', len(ids))
print('tokens/characters:', len(ids) / len(paragraph))
print('tokens/bytes:', len(ids) / len(paragraph.encode('utf-8')))

## What I learned

1. Why doesn't an LLM directly consume words?

2. Why do bytes appear before BPE in a byte-level tokenizer?

3. What does BPE actually do?

4. What is the relationship between tokens and vocabulary IDs?

5. Why can two visually similar strings have different token counts?

6. What surprised me during the experiments?

7. Which of my token-count predictions were wrong, and why?

## Topic 1 — Done When

I can predict which strings cost more tokens and explain why,
then verify those predictions using a tokenizer.

## References

- Andrej Karpathy — Let's build the GPT Tokenizer
- Andrej Karpathy — minbpe
- OpenAI Tokenizer / tiktoken documentation